# `Normalizer` - Normalisation des données agricoles

**Module :** `kadi.kidas.normalizer`  
**Classe :** `Normalizer`

---

## Introduction

`Normalizer` harmonise les données agricoles vers des standards internationaux et Bénin :

- **Noms de colonnes** : conversion en `snake_case` sans accents (`Température Min` → `temperature_min`)
- **Unités AgriTech** : conversions tonnes ↔ kg, cm ↔ m, litres ↔ m³, etc.
- **Noms de cultures** : harmonisation vers la nomenclature **FAO** (`maïs` → `maize`, `niébé` → `cowpea`)
- **Noms de marchés** : correspondance avec les marchés officiels **Bénin** + enrichissement GPS
- **Devises** : conversion XOF → USD/EUR via taux de change
- **Géométrie Shapely** : création d'une colonne `geometry` (Point) pour intégration SIG

## 1. Données de démonstration

In [ ]:
import os
import pandas as pd

import numpy as np
import pandas as pd

from kadi.kidas.normalizer import Normalizer

# DataFrame avec noms non normalisés (tel qu'extrait d'un fichier Excel)
df_brut = pd.DataFrame({
    'Température (°C)':   [28.5, 30.1, 26.8, 32.0],
    'Précipitation (mm)': [120.0, 85.0, 200.0, 45.0],
    'Rendement (T/ha)':   [1.5, 0.8, 2.0, 1.2],   # en tonnes, à convertir en kg
    'culture':            ['Maïs', 'Niébé', 'igname', 'SORGHO'],
    'marche':             ['Dantokpa Cotonou', 'Parakou marche central', 'Bohicon', 'Kandi'],
    'prix_XOF':           [350, 500, 250, 300],
    'lat':                [6.366, 9.337, 7.181, 11.133],
    'lon':                [2.437, 2.629, 2.067, 2.740],
})

print(f"Colonnes brutes : {list(df_brut.columns)}")
df_brut

## 2. Initialisation de `Normalizer`

In [ ]:
normalizer = Normalizer(df_brut)
print(repr(normalizer))

## 3. `norm_cols()` - Conversion en `snake_case`

In [ ]:
normalizer = Normalizer(df_brut)
df_colonnes = normalizer.norm_cols()

print("Correspondance colonnes :")
for ancienne, nouvelle in zip(df_brut.columns, df_colonnes.columns):
    indicateur = '→' if ancienne != nouvelle else '='
    print(f"  {ancienne:25s} {indicateur} {nouvelle}")

## 4. `convert_units()` - Conversion des unités AgriTech

In [ ]:
# Dictionnaire de conversion : colonne → unité source
# Les valeurs sont converties vers l'unité standard (kg, m, m³...)
conversions = {
    'Rendement (T/ha)': 'tonne',    # tonnes → kg (× 1000)
}

normalizer = Normalizer(df_brut)
df_unites = normalizer.convert_units(unit_map=conversions)

print("Conversion tonnes → kg :")
for avant, apres in zip(df_brut['Rendement (T/ha)'], df_unites['Rendement (T/ha)']):
    print(f"  {avant:.1f} T/ha → {apres:.0f} kg/ha")

In [ ]:
df_test_unites = pd.DataFrame({
    "production_t": [1.5, 2.0, 0.75],
    "sacs": [4, 3, 2],
})
normalizer = Normalizer(df_test_unites)
df_conv = normalizer.convert_units({"production_t": "tonne", "sacs": "sac_50kg"})
print(df_conv)

## 5. `std_crops()` - Harmonisation vers la nomenclature FAO

In [ ]:
print("Noms de cultures bruts :")
print(df_brut['culture'].tolist())

normalizer = Normalizer(df_brut)
df_cultures = normalizer.std_crops(col='culture')

print("\nAprès normalisation FAO :")
print(df_cultures['culture'].tolist())

In [ ]:
# Démonstration étendue du dictionnaire FAO
df_cultures_variees = pd.DataFrame({
    'culture': [
        'Maïs', 'mais', 'MAÏS',         # → maize
        'Niébé', 'niebe', 'haricot',     # → cowpea
        'Igname', 'igname de Bénin',     # → yam
        'Manioc', 'Cassava', 'cassave',  # → cassava
        'Sorgho', 'petit mil',           # → sorghum
        'Arachide', 'GROUNDNUT',         # → groundnut
    ]
})

norm = Normalizer(df_cultures_variees)
df_fao = norm.std_crops(col='culture')

comparaison = pd.DataFrame({
    'Brut': df_cultures_variees['culture'],
    'FAO':  df_fao['culture'],
})
print(comparaison.to_string(index=False))

## 6. `std_markets()` - Correspondance avec les marchés officiels Bénin

In [ ]:
print("Noms de marchés bruts :")
print(df_brut['marche'].tolist())

normalizer = Normalizer(df_brut)
df_marches = normalizer.std_markets(col='marche')

print("\nMarchés normalisés (+ coordonnées GPS) :")
colonnes_aff = ['marche', 'market_lat', 'market_lon']
cols_presentes = [c for c in colonnes_aff if c in df_marches.columns]
print(df_marches[cols_presentes].to_string(index=False))

## 7. Conversion des devises avec `convert_currency()`

La méthode convertit la colonne fournie sur place et enregistre le taux indicatif dans `mappings()`.

In [ ]:
normalizer = Normalizer(df_brut)

# Le taux illustratif interne vaut environ 600 XOF pour 1 USD.
df_devise = normalizer.convert_currency(col="prix_XOF", from_="XOF", to="USD")
print(df_devise["prix_XOF"].rename("prix_USD"))

## 8. `std_coords()` - Création d'une colonne Shapely Point

In [ ]:
try:
    from shapely.geometry import Point
    SHAPELY_DISPONIBLE = True
except ImportError:
    SHAPELY_DISPONIBLE = False
    print("Note : shapely non installé - pip install shapely")

if SHAPELY_DISPONIBLE:
    normalizer = Normalizer(df_brut)
    df_geo = normalizer.std_coords(lat='lat', lon='lon')

    print("Colonne 'geometry' créée :")
    for idx, geom in df_geo['geometry'].items():
        print(f"  Ligne {idx} : {geom} (type={geom.geom_type})")

## 9. Pipeline de normalisation complet

In [ ]:
normalizer = Normalizer(df_brut)

# Étape 1 : Normalisation des noms de colonnes
normalizer.norm_cols()

# Étape 2 : Conversion des unités
normalizer.convert_units({'rendement_t_ha': 'tonne'})

# Étape 3 : Cultures FAO
normalizer.std_crops(col='culture')

# Étape 4 : Marchés Bénin
normalizer.std_markets(col='marche')

print("DataFrame normalisé :")
print(f"  Colonnes : {list(normalizer.df.columns)}")
normalizer.df

## 10. `mappings()` - Journal de normalisation

In [ ]:
mapping = normalizer.mappings()

print("=== Rapport de Normalisation ===")
for categorie, details in mapping.items():
    if details:
        print(f"\n{categorie.upper()} :")
        if isinstance(details, dict):
            for cle, val in details.items():
                print(f"  {cle:25s} → {val}")
        elif isinstance(details, list):
            for item in details:
                print(f"  - {item}")

## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `norm_cols()` | snake_case, suppression accents et caractères spéciaux |
| `convert_units(unit_map)` | Conversion d’unités de masse (tonne, gramme, sac...) vers le kilogramme |
| `std_crops(col)` | Nomenclature FAO (maïs→maize, niébé→cowpea...) |
| `std_markets(col)` | Marchés officiels Bénin + enrichissement GPS |
| `convert_currency(col, from, to, taux)` | Conversion XOF/USD/EUR |
| `std_coords(lat_col, lon_col)` | Création colonne Shapely Point |
| `mappings()` | Journal de toutes les transformations effectuées |